# Notebook 4: Variance Reduction Techniques
**Project:** Monte Carlo Risk & Derivatives Pricing
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations')


Loaded 2,609 observations


In [2]:
from options_pricing import bs_call
from variance_reduction import mc_naive, mc_antithetic
S0,K,r,T,sigma = 100,100,0.05,1.0,0.20
bs_true = bs_call(S0,K,T,r,sigma)
sizes=[500,1000,2000,5000,10000,20000,50000]
naive_se,anti_se=[],[]
for n in sizes:
    _,se_n=mc_naive(S0,K,T,r,sigma,n)
    _,se_a=mc_antithetic(S0,K,T,r,sigma,n)
    naive_se.append(se_n); anti_se.append(se_a)
reduction=[(n-a)/n*100 for n,a in zip(naive_se,anti_se)]
print('Standard Error Comparison:')
for s,n,a,r in zip(sizes,naive_se,anti_se,reduction):
    print(f'  n={s:6d}: Naive={n:.5f}  Antithetic={a:.5f}  Reduction={r:.1f}%')

Standard Error Comparison:
  n=   500: Naive=0.71151  Antithetic=0.67786  Reduction=4.7%
  n=  1000: Naive=0.49696  Antithetic=0.48246  Reduction=2.9%
  n=  2000: Naive=0.35449  Antithetic=0.33873  Reduction=4.4%
  n=  5000: Naive=0.21818  Antithetic=0.21506  Reduction=1.4%
  n= 10000: Naive=0.15538  Antithetic=0.15434  Reduction=0.7%
  n= 20000: Naive=0.11027  Antithetic=0.10997  Reduction=0.3%
  n= 50000: Naive=0.06905  Antithetic=0.06914  Reduction=-0.1%


In [3]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
ax1=axes[0]
ax1.plot(sizes,naive_se,'o-',color='#185FA5',linewidth=2,markersize=6,label='Naive MC')
ax1.plot(sizes,anti_se,'s-',color='#E24B4A',linewidth=2,markersize=6,label='Antithetic Variates')
ax1.set_title('Standard Error: Naive vs Antithetic Variates')
ax1.set_xlabel('Simulations'); ax1.set_ylabel('Standard Error')
ax1.set_xscale('log'); ax1.legend(fontsize=9)
ax2=axes[1]
bars=ax2.bar(range(len(sizes)),reduction,color='#1D9E75',alpha=0.85,edgecolor='white')
ax2.set_xticks(range(len(sizes))); ax2.set_xticklabels([str(s) for s in sizes],rotation=30,fontsize=9)
ax2.set_title('SE Reduction from Antithetic Variates (%)'); ax2.set_ylabel('SE Reduction (%)')
for i,v in enumerate(reduction):
    ax2.text(i,v+0.3,f'{v:.1f}%',ha='center',va='bottom',fontsize=9)
plt.tight_layout(); plt.savefig('../results/05_variance_reduction.png',dpi=150,bbox_inches='tight')
plt.show()